# VoxConverse test seti değerlendirmesi ve hız ölçümü

Sistem, geliştirme sırasında **hiç kullanılmamış** VoxConverse **test** setinde (sürüm 0.3, 232 kayıt) değerlendirilir. Yeni algoritma yoktur: deneyler `src/toplu.py`, hat `src/main.py` ile çalışır; yardımcılar `src/test_seti.py`'dedir.

**Test setinde hiçbir ayar yapılmaz.** E2 ve E4 eşikleri, MFCC ölçeği ve E5 eşiği `src/varsayilanlar.json`'dan, E5 değişim modeli Drive'daki `Hafta4/modeller/degisim_modeli.joblib`'den **aynen** alınır (hepsi dev setinin eğitim yarısında belirlendi). Test sonuçlarına bakarak eşik veya model değiştirilmez.

| Bölüm | İş | Tahmini süre (GPU / CPU) |
|---|---|---|
| 0–1 | Ayarlar, kurulum | 3–5 dk |
| 2 | Test zip'i (4,27 GB): Drive'da varsa kopya, yoksa indirme + doğrulama + Drive'a kopya | 5–10 dk / ilk kez 15–40 dk |
| 4 | Önbellek: VAD + MFCC + ECAPA (~43 saat ses) | ~40–70 dk / ~2–3 sa |
| 5–6 | E2, E4, E5 ve özet | ~10–20 dk |
| 7 | Hız ölçümü (90 sn kayıt, MFCC ve ECAPA, 1 ısınma + 5 ölçüm) | 1–3 dk |

Her bölüm **kaldığı yerden devam eder** (önbellek `Hafta7/onbellek_test`, sonuçlar `Hafta7/sonuclar/test`). Oturum düşerse 0. ve 1. bölümü çalıştırıp devam edin. Hız ölçümünü (7. bölüm) GPU ve CPU çalışma zamanında ayrı ayrı çalıştırabilirsiniz; sonuçlar cihaz adıyla ayrı dosyalara yazılır ve birikir.

## 0. Ayarlar

In [ ]:
DAL = "main"                                       # çalıştırılacak repo dalı

import os, sys, json, subprocess, shutil, time

KOK = "/content/drive/MyDrive/BilgisayarTasarim_Proje68"
REPO_URL = "https://github.com/HekimSefkan/Konusmaci-Tespiti.git"
REPO = "/content/Konusmaci-Tespiti"

# Test zip'i kaynak sırası: (1) Drive, (2) KAIST resmi yansısı, (3) Oxford, (4) Hugging Face yansısı.
# Zip kaynakları MD5 ile doğrulanır (KAIST sayfasındaki resmi değer).
KT_TEST_ZIP_URL = os.environ.get("KT_TEST_ZIP_URL", "https://mm.kaist.ac.kr/datasets/voxconverse/data/voxconverse_test_wav.zip")
OXFORD_ZIP_URL = "https://www.robots.ox.ac.uk/~vgg/data/voxconverse/data/voxconverse_test_wav.zip"
TEST_ZIP_MD5 = "834558bbd9b1ffd2d4893181556ceddd"
HF_REPO = "ggfox00000/dia-voxconverse-test"                # son çare: audio/test/*.wav
ZIP_DRIVE = f"{KOK}/veri_kaynak/voxconverse_test_wav.zip"  # resmi zip bir kez indirilip burada saklanır
HF_ZIP_DRIVE = f"{KOK}/veri_kaynak/voxconverse_test_wav_hf_yansi.zip"  # HF yansısından toplanan zip
ZIP_YEREL = "/content/veri/voxconverse_test_wav.zip"      # hız için Colab yerel diski
VOX_URL = "https://github.com/joonson/voxconverse"         # master = sürüm 0.3
VOX_REPO = "/content/voxconverse"
VOX_TEST = f"{VOX_REPO}/test"                              # test referans RTTM'leri

MODEL = f"{KOK}/Hafta4/modeller/degisim_modeli.joblib"     # E5 (dev eğitim yarısında eğitildi)
ECAPA_ONBELLEK = f"{KOK}/Hafta4/modeller/ecapa"
DEV_OZET = f"{KOK}/Hafta4/sonuclar/ozet.csv"               # dev değerlendirme yarısı (108 kayıt)
DEV_E5 = f"{KOK}/Hafta5/deney/E5_ayarli_ozet.json"         # E5, eşik 0,70 (varsa)
HIZ_SES = f"{KOK}/Hafta2/ornek_ses/sldwj_000.0-090.0.wav"  # 90 sn hız ölçümü kaydı

ONBELLEK = f"{KOK}/Hafta7/onbellek_test"
SONUC = f"{KOK}/Hafta7/sonuclar/test"
DENEY_DIZIN = f"{SONUC}/deneyler"                          # toplu.py çıktıları (CSV, RTTM, ozet.csv)
LISTE = f"{SONUC}/test_listesi.txt"

os.environ["KT_MODEL_ONBELLEK"] = ECAPA_ONBELLEK

## 1. Kurulum

Drive bağlanır, repo klonlanır ya da uzak daldaki son sürüme güncellenir (`git fetch` + `git reset --hard`). `requirements.txt`'teki `torch`, `torchaudio` ve CPU `--extra-index-url` satırları atlanır; Colab'ın torch'u kullanılır (Hafta 4 defterindeki gibi). torchaudio ≥ 2.9 ile speechbrain 1.0.3 uyumsuzluğu `src/uyumluluk.py` yamasıyla giderilir. Önceden yüklü paketler değişirse hücre çalışma zamanını yeniden başlatmanızı ister; sonra 0. ve 1. bölümü tekrar çalıştırın (kurulum atlanır).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

if not os.path.isdir(f"{REPO}/.git"):
    subprocess.run(["git", "clone", "--branch", DAL, REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "fetch", "origin", DAL], check=True)
    subprocess.run(["git", "-C", REPO, "checkout", DAL], check=True)
    subprocess.run(["git", "-C", REPO, "reset", "--hard", f"origin/{DAL}"], check=True)
print("Repo:", subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())

PAKETLER = ["numpy", "scipy", "scikit-learn", "torch", "torchaudio", "speechbrain", "silero-vad", "onnxruntime",
            "librosa", "soundfile", "pyannote.metrics", "huggingface_hub", "joblib", "matplotlib"]


def surumler():
    """Paket sürümleri, ayrı bir süreçte (bu çekirdekte yüklü eski sürümler karışmasın)."""
    k = ("import importlib.metadata as m, json\nd = {}\n"
         f"for p in {PAKETLER!r}:\n"
         "    try: d[p] = m.version(p)\n    except Exception: d[p] = None\nprint(json.dumps(d))")
    return json.loads(subprocess.run([sys.executable, "-c", k], capture_output=True, text=True).stdout)


ISARET = "/content/.kt_test_kurulum_tamam"
if os.path.isfile(ISARET):
    print("Kurulum daha önce yapılmış; atlanıyor.")
else:
    once = surumler()
    secilen = []
    for s in open(f"{REPO}/requirements.txt", encoding="utf-8"):
        s = s.strip()
        if not s or s.startswith("#") or s.startswith("--extra-index-url"):
            continue
        if s.startswith(("torch==", "torchaudio==")):  # Colab'ın torch'u kullanılır
            continue
        secilen.append(s)
    open("/content/gereksinimler_test.txt", "w").write("\n".join(secilen) + "\n")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "/content/gereksinimler_test.txt"], check=True)
    sonra = surumler()
    degisen = {p: (once[p], sonra[p]) for p in PAKETLER if once[p] != sonra[p]}
    json.dump({"surumler": sonra}, open(ISARET, "w"))
    onceden_yuklu = [p for p, (a, b) in degisen.items() if a is not None]
    if onceden_yuklu:
        print("!" * 78)
        print("!!  ÇALIŞMA ZAMANINI YENİDEN BAŞLATIN: Çalışma zamanı > Oturumu yeniden başlat")
        print("!!  Önceden yüklü paketler değişti:", ", ".join(onceden_yuklu))
        print("!!  Sonra 0. ve 1. bölümü tekrar çalıştırın (kurulum atlanır).")
        print("!" * 78)

print("Sürümler:")
for p, v in surumler().items():
    print(f"  {p:<18} {v}")
r = subprocess.run([sys.executable, "-c",
                    "import torch; print('CUDA:', torch.cuda.is_available()); "
                    "from src.uyumluluk import speechbrain_hazirla; print('uyumluluk yaması:', speechbrain_hazirla() or 'gerekmedi'); "
                    "from speechbrain.inference.speaker import EncoderClassifier; print('speechbrain: içe aktarıldı')"],
                   cwd=REPO, capture_output=True, text=True)
print(r.stdout.strip())
if r.returncode != 0:
    print(*r.stderr.strip().splitlines()[-5:], sep="\n")
    print("UYARI: speechbrain içe aktarılamadı; ECAPA deneyleri çalışmaz.")

os.chdir(REPO)
sys.path.insert(0, REPO)

## 2. Veri: test zip'i ve referans RTTM'leri

Kaynak sırası:
1. Drive'daki `veri_kaynak/voxconverse_test_wav.zip` (yerel diske kopyalanır, MD5 doğrulanır),
2. KAIST resmi yansısı (`KT_TEST_ZIP_URL`, 0. bölümde değiştirilebilir),
3. Oxford adresi,
4. son çare Hugging Face yansısı `ggfox00000/dia-voxconverse-test` (yalnızca `audio/test/*.wav`; WAV'lar Colab yerel diskinde sıkıştırmasız bir zip'e toplanır).

Zip kaynakları resmi MD5 (`834558bbd9b1ffd2d4893181556ceddd`) ile doğrulanır; adreslerde önce `HEAD`, başarısızsa `GET` denenir, 401/403'te sıradaki kaynağa geçilir; indirme `wget -c` ile kaldığı yerden sürer. Resmi zip Drive'da yoksa Drive'a kopyalanır (sonraki oturumda yeniden indirilmesin); Drive'daki MD5'i tutmayan dosyanın üzerine yazılmaz. HF yansısından toplanan zip Drive'a ayrı adla (`voxconverse_test_wav_hf_yansi.zip`) kaydedilir. **Etiketler her durumda** resmi GitHub deposunun `test/` klasöründen (master = v0.3); ses adları RTTM adlarıyla eşleştirilir, eksik/fazla raporlanır. Kullanılan kaynak ve MD5 sonucu `kaynak.json`'a yazılır.

In [ ]:
import zipfile
from src import test_seti

# Etiketler her durumda resmi GitHub deposunun test/ klasöründen (master = v0.3)
if not os.path.isdir(f"{VOX_REPO}/.git"):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "master", VOX_URL, VOX_REPO], check=True)
print("VoxConverse:", subprocess.run(["git", "-C", VOX_REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())
TEST_RTTM = test_seti.rttm_kayitlari(VOX_TEST)
print(len(TEST_RTTM), "test RTTM")

# (1) Drive, (2) KAIST, (3) Oxford — MD5 doğrulamalı
KAYNAK = test_seti.zip_hazirla(ZIP_YEREL, ZIP_DRIVE, [KT_TEST_ZIP_URL, OXFORD_ZIP_URL], TEST_ZIP_MD5)

if KAYNAK["kaynak"] is None:
    # (4) son çare: Hugging Face yansısı (tek tek WAV)
    print("Resmi zip alınamadı; Hugging Face yansısına geçiliyor:", HF_REPO)
    if os.path.isfile(HF_ZIP_DRIVE):
        print("Drive'da HF yansısından toplanmış zip var; yerel diske kopyalanıyor...")
        shutil.copyfile(HF_ZIP_DRIVE, ZIP_YEREL)
    else:
        from huggingface_hub import snapshot_download

        hf_dizin = snapshot_download(HF_REPO, repo_type="dataset", allow_patterns=["audio/test/*.wav"],
                                     local_dir="/content/hf_voxconverse_test")
        wavlar = test_seti.klasor_wavlari(os.path.join(hf_dizin, "audio", "test"))
        if not wavlar:
            raise RuntimeError(f"HF yansısında audio/test/*.wav bulunamadı: {HF_REPO}")
        e = test_seti.eslestir(wavlar, TEST_RTTM)
        print(f"HF yansısı: {len(wavlar)} WAV, resmi RTTM ile ortak {len(e['ortak'])}")
        print(f"  eksik (RTTM var, WAV yok) {len(e['yalniz_rttm'])}:", e["yalniz_rttm"] or "yok")
        print(f"  fazla (WAV var, RTTM yok) {len(e['yalniz_ses'])}:", e["yalniz_ses"] or "yok")
        print("WAV'lar sıkıştırmasız (ZIP_STORED) zip'e toplanıyor (toplu.py zip'ten okur)...")
        test_seti.wav_zip_olustur(wavlar, ZIP_YEREL)
        os.makedirs(os.path.dirname(HF_ZIP_DRIVE), exist_ok=True)
        gecici = HF_ZIP_DRIVE + ".kopyalaniyor"
        shutil.copyfile(ZIP_YEREL, gecici)
        os.replace(gecici, HF_ZIP_DRIVE)
    KAYNAK = {**KAYNAK, "kaynak": f"hf:{HF_REPO}", "md5": None, "md5_dogru": None,
              "boyut": os.path.getsize(ZIP_YEREL), "not": "Hugging Face yansısı (tek tek WAV); resmi zip MD5'i uygulanamaz"}
elif not os.path.isfile(ZIP_DRIVE):
    print("Drive'a kopyalanıyor (sonraki oturumda yeniden indirilmesin)...")
    os.makedirs(os.path.dirname(ZIP_DRIVE), exist_ok=True)
    gecici = ZIP_DRIVE + ".kopyalaniyor"
    shutil.copyfile(ZIP_YEREL, gecici)
    os.replace(gecici, ZIP_DRIVE)
elif KAYNAK["kaynak"] != "drive":
    print("UYARI: Drive'daki zip'in MD5'i tutmadı; üzerine yazılmadı, elle kontrol edin:", ZIP_DRIVE)

print(test_seti.kaynak_metni(KAYNAK))
for d in KAYNAK.get("denemeler", []):
    print("  ", d)
os.makedirs(SONUC, exist_ok=True)
test_seti.kaynak_yaz(SONUC, KAYNAK)

ESLESME = test_seti.test_listesi_yaz(ZIP_YEREL, VOX_TEST, LISTE)
json.dump(ESLESME, open(f"{SONUC}/eslesme.json", "w"), ensure_ascii=False, indent=1)
print(f"Zip'te ses: {len(ESLESME['ortak']) + len(ESLESME['yalniz_ses'])}, test RTTM: "
      f"{len(ESLESME['ortak']) + len(ESLESME['yalniz_rttm'])}, değerlendirilecek (ortak): {len(ESLESME['ortak'])}")
print("Yalnız seste olanlar:", ESLESME["yalniz_ses"] or "yok")
print("Yalnız RTTM'de olanlar:", ESLESME["yalniz_rttm"] or "yok")

## 3. Sabit ayarlar (test setinde değiştirilmez)

E2 ve E4 eşikleri, MFCC ölçeği ve E5 eşiği `src/varsayilanlar.json`'dan, E5 modeli Drive'dan okunur ve `ayarlar.json`'a (commit ve model özetiyle) kaydedilir. Bu değerler dev setinin eğitim yarısında belirlendi; test sonuçlarına göre **değiştirilmez**.

In [ ]:
import hashlib
from src import varsayilanlar, toplu

v = varsayilanlar.yukle()
AYARLAR = {
    "E2": {"yontem": "mfcc", "esik": varsayilanlar.esik("mfcc"), "mfcc_olcek": "varsayilanlar.json"},
    "E4": {"yontem": "ecapa", "esik": varsayilanlar.esik("ecapa")},
    "E5": {"yontem": "ecapa + değişim modeli", "esik": float(v["e5_esik"]), "model": MODEL},
}
if not os.path.isfile(MODEL):
    raise FileNotFoundError(f"E5 modeli yok: {MODEL}")
model_ozet = hashlib.sha256(open(MODEL, "rb").read()).hexdigest()
commit = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
json.dump({"ayarlar": AYARLAR, "kaynak": v.get("kaynak"), "e5_esik_kaynak": v.get("e5_esik_kaynak"),
           "model_sha256": model_ozet, "commit": commit, "tarih": time.strftime("%Y-%m-%dT%H:%M:%S")},
          open(f"{SONUC}/ayarlar.json", "w"), ensure_ascii=False, indent=1)
for d, a in AYARLAR.items():
    print(d, a)
print("Model SHA-256:", model_ozet[:16], "... | commit:", commit[:8])

## 4. Önbellek: VAD + MFCC + ECAPA

Her kaydın VAD bölgeleri ve pencere vektörleri `Hafta7/onbellek_test` altına yazılır; hücre kesilirse yeniden çalıştırın, yapılmış kayıtlar atlanır. Hata veren kayıtlar `hata_gunlugu.txt`'ye yazılır.

In [ ]:
KAYITLAR = toplu.listeleri_oku([LISTE])
print(len(KAYITLAR), "kayıt")
toplu.onbellek_doldur(ZIP_YEREL, KAYITLAR, ONBELLEK, ("mfcc", "ecapa"), gunluk_dizin=SONUC)

## 5. Deneyler: E2 (MFCC), E4 (ECAPA, ana sistem), E5 (değişim modeli)

Sonuçlar `deneyler/` altında: kayıt başına CSV, RTTM'ler ve `ozet.csv`. Yapılmış kayıtlar atlanır.

In [ ]:
for deney in ("E2", "E4", "E5"):
    a = AYARLAR[deney]
    toplu.deney_calistir(ZIP_YEREL, VOX_TEST, KAYITLAR, deney, DENEY_DIZIN, ONBELLEK, esik=a["esik"],
                         model_yolu=MODEL if deney == "E5" else None,
                         mfcc_olcek=varsayilanlar.mfcc_olcek() if deney == "E2" else None)
    print()

## 6. Özet

- `ozet.csv`: dev değerlendirme yarısı (Hafta 4; E5 için varsa Hafta 5'in 0,70 eşikli ölçümü) ve test yan yana.
- `gruplar_E4.csv`: referans konuşmacı sayısına göre (1, 2, 3–4, 5+) E4 DER, JER, F1.
- `en_kotu_E4.csv`: E4'ün DER'e göre en kötü 10 kaydı.
- `deneyler.png`: test deneylerinin DER / JER / F1 grafiği (`src/gorsel`).

In [ ]:
from IPython.display import Image, display
from src import bolme, gorsel

if not os.path.isfile(DEV_E5):
    print("UYARI: Hafta 5 E5 (0,70) dev ölçümü bulunamadı; E5'in dev değeri Hafta 4 ozet.csv'den (eşik 0,79346).")
KARSILASTIRMA = test_seti.karsilastirma(DEV_OZET, f"{DENEY_DIZIN}/ozet.csv", dev_ek={"E5": DEV_E5})
test_seti._csv_yaz(f"{SONUC}/ozet.csv", KARSILASTIRMA, test_seti.KARSILASTIRMA_SUTUNLARI)

SAYILAR = bolme.konusmaci_sayilari(VOX_TEST)
e4 = toplu._csv_oku(f"{DENEY_DIZIN}/E4.csv")
test_seti._csv_yaz(f"{SONUC}/gruplar_E4.csv", test_seti.grup_tablosu(e4, SAYILAR), test_seti.GRUP_SUTUNLARI)
test_seti._csv_yaz(f"{SONUC}/en_kotu_E4.csv", test_seti.en_kotu(e4, 10, SAYILAR, f"{DENEY_DIZIN}/E4"),
                   test_seti.EN_KOTU_SUTUNLARI)
gorsel.sonuc_grafigi(f"{DENEY_DIZIN}/ozet.csv", f"{SONUC}/deneyler.png")

print(test_seti.tablo_metni(KARSILASTIRMA, test_seti.KARSILASTIRMA_SUTUNLARI))
print()
print(test_seti.tablo_metni(test_seti.grup_tablosu(e4, SAYILAR), test_seti.GRUP_SUTUNLARI))
print()
print(test_seti.tablo_metni(test_seti.en_kotu(e4, 10, SAYILAR, f"{DENEY_DIZIN}/E4"), test_seti.EN_KOTU_SUTUNLARI))
display(Image(f"{SONUC}/deneyler.png"))

## 7. Hız ölçümü

Tek 90 sn kayıt (`Hafta2/ornek_ses/sldwj_000.0-090.0.wav`); modeller ölçümden önce bir kez yüklenir; 1 ısınma + 5 ölçüm, medyan. Aşamalar: yükleme (ses okuma), VAD, öznitelik, kümeleme. MFCC ve ECAPA ayrı. Hafta 2 hedefi: **90 sn ses ≤ 45 sn**. Sonuç cihaz adıyla `hiz/hiz_<cihaz>.json`'a yazılır; GPU ve CPU çalışma zamanında ayrı ayrı çalıştırılırsa ikisi de birikir (CPU için: *Çalışma zamanı türünü değiştir → CPU*, sonra 0, 1 ve 7. bölüm).

In [ ]:
from src import oznitelik, vad

vad._silero_modeli()
oznitelik.ecapa_modeli()  # modeller bir kez yüklenir; ölçüme girmez
CIHAZ = test_seti.cihaz_bilgisi()
print("Cihaz:", CIHAZ)
HIZ = [test_seti.hiz_olc(HIZ_SES, yontem, tekrar=5, isinma=1) for yontem in ("mfcc", "ecapa")]
print("Yazıldı:", test_seti.hiz_yaz(HIZ, f"{SONUC}/hiz", CIHAZ))
print(test_seti.hiz_tablosu(f"{SONUC}/hiz"))

## 8. Kopyalanacak özet

Aşağıdaki çıktının tamamını (en fazla ~60 satır) kopyalayıp paylaşın.

In [ ]:
ESLESME = json.load(open(f"{SONUC}/eslesme.json", encoding="utf-8")) if os.path.isfile(f"{SONUC}/eslesme.json") else None
print(test_seti.kisa_ozet(SONUC, ESLESME))